## Autoencoder Workflow
Load preprocessed FIF hypnograms, train an LSTM autoencoder, and cluster the latent embeddings.


In [1]:

# %%
import glob
import os
import random
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import umap
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


## Imports and Dependencies
Imports for training, clustering, and plotting.


In [ ]:

# Convert annotations to fixed-length epoch sequences
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


class HypnogramDataset(Dataset):
    def __init__(self, fif_root, cohort_mode="sc_only", preprocessed_dir="preprocessed", max_len=1024, epoch_sec=30):
        pattern = os.path.join(fif_root, "**", preprocessed_dir, "*.fif")
        all_files = sorted(glob.glob(pattern, recursive=True))
        self.files = self._filter_files(all_files, cohort_mode)
        if not self.files:
            raise ValueError(f"No FIF files found for cohort_mode={cohort_mode!r} under {fif_root}.")

        self.max_len = max_len
        self.epoch_sec = epoch_sec
        self.cohort_mode = cohort_mode
        self.data = []
        self.cohorts = []
        self.subject_ids = []
        self.file_paths = []
        self.truncated_flags = []

        self._process_files()

    @staticmethod
    def _filter_files(files, cohort_mode):
        if cohort_mode == "sc_only":
            return [file for file in files if "sleep-cassette" in file]
        if cohort_mode == "st_only":
            return [file for file in files if "sleep-telemetry" in file]
        if cohort_mode == "both":
            return files
        raise ValueError("cohort_mode must be 'sc_only', 'st_only', or 'both'.")

    @staticmethod
    def _extract_subject_id(file_path):
        filename = os.path.basename(file_path)
        match = re.match(r"(SC4\d{2}|ST7\d{2})", filename)
        if match:
            return match.group(1)
        raise ValueError(f"Could not extract a subject ID from {filename}.")

    def _process_files(self):
        for file in tqdm(self.files, desc="Loading FIF files for PyTorch"):
            raw = mne.io.read_raw_fif(file, preload=False, verbose="ERROR")

            total_duration = raw.times[-1]
            num_epochs = int(total_duration // self.epoch_sec)
            sequence = np.full(num_epochs, -1, dtype=np.int64)

            for annot in raw.annotations:
                start_ep = int(annot["onset"] // self.epoch_sec)
                end_ep = int((annot["onset"] + annot["duration"]) // self.epoch_sec)
                stage = int(annot["description"])
                if stage != -1:
                    sequence[start_ep:end_ep] = stage

            was_truncated = len(sequence) > self.max_len
            if was_truncated:
                sequence = sequence[: self.max_len]
            else:
                pad_width = self.max_len - len(sequence)
                sequence = np.pad(sequence, (0, pad_width), constant_values=-1)

            self.data.append(sequence)
            self.cohorts.append(0 if "sleep-cassette" in file else 1)
            self.subject_ids.append(self._extract_subject_id(file))
            self.file_paths.append(file)
            self.truncated_flags.append(was_truncated)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = torch.tensor(self.data[idx], dtype=torch.long)
        cohort = torch.tensor(self.cohorts[idx], dtype=torch.long)

        seq_onehot = F.one_hot(torch.where(seq == -1, 5, seq), num_classes=6).float()
        seq_onehot = seq_onehot.permute(1, 0)

        return seq_onehot, seq, cohort


fif_root = os.path.join(".", "sleep-edf-database-expanded-1.0.0")
healthy_dataset = HypnogramDataset(fif_root, cohort_mode="sc_only")
dataset = healthy_dataset

try:
    unhealthy_dataset = HypnogramDataset(
        fif_root,
        cohort_mode="st_only",
        max_len=healthy_dataset.max_len,
        epoch_sec=healthy_dataset.epoch_sec,
    )
except ValueError:
    unhealthy_dataset = None

subject_ids = np.array(healthy_dataset.subject_ids)
indices = np.arange(len(healthy_dataset))
splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=SEED)
train_indices, val_indices = next(splitter.split(indices, groups=subject_ids))

train_dataset = Subset(healthy_dataset, train_indices)
val_dataset = Subset(healthy_dataset, val_indices)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_subjects = len(np.unique(subject_ids))
train_subjects = len(np.unique(subject_ids[train_indices]))
val_subjects = len(np.unique(subject_ids[val_indices]))
num_truncated = int(np.sum(healthy_dataset.truncated_flags))

print(f"Healthy nights loaded: {len(healthy_dataset)} from {num_subjects} subjects.")
print(f"Train nights: {len(train_indices)} across {train_subjects} healthy subjects.")
print(f"Validation nights: {len(val_indices)} across {val_subjects} healthy subjects.")
print(f"Healthy sequences truncated to {healthy_dataset.max_len} epochs: {num_truncated}.")
if unhealthy_dataset is not None:
    unhealthy_subjects = len(np.unique(unhealthy_dataset.subject_ids))
    unhealthy_truncated = int(np.sum(unhealthy_dataset.truncated_flags))
    print(f"Unhealthy nights available for projection: {len(unhealthy_dataset)} from {unhealthy_subjects} subjects.")
    print(f"Unhealthy sequences truncated to {unhealthy_dataset.max_len} epochs: {unhealthy_truncated}.")


Loading FIF files for PyTorch:  48%|████▊     | 21/44 [02:34<02:42,  7.04s/it]


## Dataset Preparation
Build fixed-length hypnogram sequences from preprocessed FIF files.

Default behavior:
- train and validate on `sleep-cassette` only
- keep `sleep-telemetry` separate and project it into the healthy latent space after training
- split healthy data by subject, not by night
- use the existing `preprocessed` FIF files, which already have wake trimmed at both ends


## Model Definition
Define the LSTM autoencoder used for sequence reconstruction.


In [ ]:
class HypnoLSTMAutoencoder(nn.Module):
    def __init__(self, seq_len=1024, num_classes=6, latent_dim=16, hidden_dim=64):
        super().__init__()
        self.seq_len = seq_len
        self.num_classes = num_classes

        self.encoder_lstm = nn.LSTM(
            input_size=num_classes,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)

        self.decoder_lstm = nn.LSTM(
            input_size=latent_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.2,
        )
        self.fc_decode = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        # x shape: (batch, seq_len, num_classes)
        x = x.permute(0, 2, 1)

        packed_x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.encoder_lstm(packed_x)
        last_hidden = h_n[-1]
        z = self.fc_mu(last_hidden)

        z_repeated = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        dec_out, _ = self.decoder_lstm(z_repeated)

        reconstruction_logits = self.fc_decode(dec_out)
        reconstruction_logits = reconstruction_logits.permute(0, 2, 1)
        return reconstruction_logits, z


class HypnoCNNAutoencoder(nn.Module):
    def __init__(self, seq_len=1024, num_classes=6, latent_dim=16, channels=32):
        super().__init__()
        self.seq_len = seq_len
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.conv_output_len = seq_len // 4

        self.encoder = nn.Sequential(
            nn.Conv1d(num_classes, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv1d(channels, channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv1d(channels, channels * 2, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )
        self.fc_mu = nn.Linear((channels * 2) * self.conv_output_len, latent_dim)

        self.fc_decode = nn.Linear(latent_dim, (channels * 2) * self.conv_output_len)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(channels * 2, channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.ConvTranspose1d(channels, num_classes, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x, lengths=None):
        # x shape: (batch, num_classes, seq_len)
        encoded = self.encoder(x)
        encoded = encoded.flatten(1)
        z = self.fc_mu(encoded)

        decoded = self.fc_decode(z).view(x.size(0), -1, self.conv_output_len)
        reconstruction_logits = self.decoder(decoded)
        return reconstruction_logits, z


model_lstm = HypnoLSTMAutoencoder(seq_len=dataset.max_len, num_classes=6, latent_dim=32)
model_cnn = HypnoCNNAutoencoder(seq_len=dataset.max_len, num_classes=6, latent_dim=16)
# Choose the active model here: model_lstm or model_cnn
model = model_cnn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model.to(device)


## Training the Autoencoder
Train with cross-entropy while ignoring padding tokens.


In [ ]:

# %%
optimizer = torch.optim.Adam(model.parameters(), lr=20e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=5)
l1_lambda = 0.0


def compute_l1_penalty(module):
    penalty = torch.zeros((), device=device)
    for param in module.parameters():
        penalty = penalty + param.abs().sum()
    return penalty


num_epochs = 200
train_losses = []
val_losses = []

for epoch in tqdm(range(num_epochs), desc="Training", total=num_epochs):
    
    # --- TRAINING ---
    model.train()
    total_train_loss = 0.0

    for x_onehot, x_labels, _ in train_dataloader:
        x_onehot = x_onehot.to(device)
        x_labels = x_labels.to(device)

        # 1. Echte Länge der Sequenz berechnen (Anzahl der Labels ungleich -1)
        # MUSS auf der CPU bleiben für PyTorch Packing!
        lengths = (x_labels != -1).sum(dim=1).cpu()

        optimizer.zero_grad()

        # 2. Längen an das Modell übergeben
        logits, _ = model(x_onehot, lengths)
        
        # 3. Padding-Tokens für die Loss-Funktion auf Klasse 5 setzen
        targets = torch.where(x_labels == -1, 5, x_labels)
        loss = criterion(logits, targets)

        if l1_lambda:
            loss = loss + l1_lambda * compute_l1_penalty(model)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)
    train_losses.append(avg_train_loss)


    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0.0
    
    with torch.no_grad():
        for x_onehot, x_labels, _ in val_dataloader:
            x_onehot = x_onehot.to(device)
            x_labels = x_labels.to(device) # Das fehlte in deinem Entwurf noch

            # 1. Längen auch in der Validierung berechnen
            lengths = (x_labels != -1).sum(dim=1).cpu()

            # 2. Längen an das Modell übergeben
            logits, _ = model(x_onehot, lengths)
            
            # 3. Targets für Loss anpassen
            targets = torch.where(x_labels == -1, 5, x_labels)
            loss = criterion(logits, targets)

            if l1_lambda:
                loss = loss + l1_lambda * compute_l1_penalty(model)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_losses.append(avg_val_loss)



## Embedding Extraction and Clustering
Fit clustering and UMAP on healthy embeddings, then project the unhealthy cohort into that same learned space.


In [ ]:
# Refresh latent embeddings and cluster assignments used by the plots below.
def refresh_latent_analysis(n_clusters=2):
    def extract_embeddings(source_dataset, desc):
        dataloader = DataLoader(source_dataset, batch_size=32, shuffle=False)
        embeddings = []
        cohorts = []

        model.eval()
        with torch.no_grad():
            for x_onehot, x_labels, cohort in tqdm(dataloader, desc=desc, leave=False):
                x_onehot = x_onehot.to(device)
                x_labels = x_labels.to(device)
                lengths = (x_labels != -1).sum(dim=1).cpu()
                _, z = model(x_onehot, lengths)
                embeddings.append(z.cpu().numpy())
                cohorts.append(cohort.numpy())

        if not embeddings:
            latent_dim = model.fc_mu.out_features
            return np.empty((0, latent_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

        return np.vstack(embeddings), np.concatenate(cohorts)

    healthy_embeddings, healthy_cohorts = extract_embeddings(
        healthy_dataset,
        desc="Extracting healthy embeddings",
    )

    if healthy_embeddings.shape[0] < n_clusters:
        raise ValueError(
            f"Need at least {n_clusters} healthy nights for clustering, got {healthy_embeddings.shape[0]}."
        )

    clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
    healthy_cluster_labels = clusterer.fit_predict(healthy_embeddings)

    healthy_silhouette = np.nan
    if len(np.unique(healthy_cluster_labels)) > 1:
        healthy_silhouette = silhouette_score(healthy_embeddings, healthy_cluster_labels)

    reducer = umap.UMAP(n_components=2, random_state=SEED)
    healthy_umap = reducer.fit_transform(healthy_embeddings)

    if unhealthy_dataset is not None and len(unhealthy_dataset) > 0:
        unhealthy_embeddings, unhealthy_cohorts = extract_embeddings(
            unhealthy_dataset,
            desc="Projecting unhealthy embeddings",
        )
        unhealthy_umap = reducer.transform(unhealthy_embeddings)
    else:
        unhealthy_embeddings = np.empty((0, healthy_embeddings.shape[1]), dtype=np.float32)
        unhealthy_cohorts = np.empty((0,), dtype=np.int64)
        unhealthy_umap = np.empty((0, 2), dtype=np.float32)

    return {
        "healthy_embeddings": healthy_embeddings,
        "healthy_cohorts": healthy_cohorts,
        "healthy_cluster_labels": healthy_cluster_labels,
        "healthy_silhouette": healthy_silhouette,
        "healthy_umap": healthy_umap,
        "unhealthy_embeddings": unhealthy_embeddings,
        "unhealthy_cohorts": unhealthy_cohorts,
        "unhealthy_umap": unhealthy_umap,
    }

latent_analysis = refresh_latent_analysis()
healthy_embeddings = latent_analysis["healthy_embeddings"]
healthy_cluster_labels = latent_analysis["healthy_cluster_labels"]
healthy_umap = latent_analysis["healthy_umap"]
unhealthy_umap = latent_analysis["unhealthy_umap"]

print(f"Healthy embeddings: {healthy_embeddings.shape[0]} nights, {healthy_embeddings.shape[1]} latent dimensions.")
if np.isnan(latent_analysis["healthy_silhouette"]):
    print("Agglomerative silhouette score could not be computed with fewer than 2 clusters.")
else:
    print(f"Agglomerative silhouette score (healthy only): {latent_analysis['healthy_silhouette']:.4f}")
print(f"Unhealthy projections: {unhealthy_umap.shape[0]} nights.")


## Training Loss Visualization
Plot training and validation loss.


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o', linestyle='-', linewidth=2, markersize=4, label='Train Loss')
plt.plot(range(1, len(val_losses) + 1), val_losses, marker='s', linestyle='--', linewidth=2, markersize=4, label='Validation Loss')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('Training and Validation Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
if "refresh_latent_analysis" not in globals():
    raise RuntimeError(
        "Run the 'Embedding Extraction and Clustering' cell before plotting the phenotype overview."
    )
# Refresh embeddings/UMAP each time this cell runs so the lower plots never use stale values.
latent_analysis = refresh_latent_analysis()
healthy_cluster_labels = latent_analysis["healthy_cluster_labels"]
healthy_umap = latent_analysis["healthy_umap"]
unhealthy_umap = latent_analysis["unhealthy_umap"]
# ==========================================
# 1. Vorbereitung: Klinische Metriken berechnen
# ==========================================
# Wir berechnen die Dauer der Nächte und die %-Anteile der Schlafphasen
# für alle gesunden Patienten, um die Cluster später interpretieren zu können.
stage_percentages = []
sequence_lengths = []
for seq in healthy_dataset.data:
    # Ignoriere Padding (-1) und undefinierte Phasen (5)
    valid_seq = seq[(seq != -1) & (seq != 5)]
    seq_len = len(valid_seq)
    sequence_lengths.append(seq_len)
    
    if seq_len > 0:
        # 0: Wake, 1: N1, 2: N2, 3: N3, 4: REM
        perc_wake = (np.sum(valid_seq == 0) / seq_len) * 100
        perc_light = (np.sum((valid_seq == 1) | (valid_seq == 2)) / seq_len) * 100
        perc_deep = (np.sum(valid_seq == 3) / seq_len) * 100
        perc_rem = (np.sum(valid_seq == 4) / seq_len) * 100
    else:
        perc_wake = perc_light = perc_deep = perc_rem = 0.0
        
    stage_percentages.append([perc_wake, perc_light, perc_deep, perc_rem])
df_metrics = pd.DataFrame(stage_percentages, columns=["% Wake", "% Light (N1+N2)", "% Deep (N3)", "% REM"])
df_metrics["Cluster"] = healthy_cluster_labels
df_metrics["Night_Length"] = sequence_lengths
# ==========================================
# 2. Die neuen, wissenschaftlichen Plots
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Discovery of Sleep Phenotypes - Latent Space Analysis", fontsize=18, fontweight='bold', y=0.98)
# --- PLOT 1: Diagnostic Plot (Ist es nur die Länge?) ---
# Dieser Plot beweist dir, ob dein Modell echte Schlafmuster lernt, 
# oder nur nach Sequenzlänge sortiert (wie aktuell der Fall).
scatter1 = axes[0, 0].scatter(
    healthy_umap[:, 0], healthy_umap[:, 1], 
    c=df_metrics["Night_Length"], cmap='magma', s=30, alpha=0.8
)
axes[0, 0].set_title("UMAP Colored by True Night Length (Epochs)", fontsize=14)
axes[0, 0].set_xlabel("UMAP Dimension 1")
axes[0, 0].set_ylabel("UMAP Dimension 2")
plt.colorbar(scatter1, ax=axes[0, 0], label="Total Epochs (excluding padding)")
# --- PLOT 2: Die Phänotypen (Cluster) ---
sns.scatterplot(
    x=healthy_umap[:, 0], y=healthy_umap[:, 1],
    hue=healthy_cluster_labels, palette="tab10",
    s=50, alpha=0.9, ax=axes[0, 1]
)
axes[0, 1].set_title("Identified Sleep Phenotypes (Agglomerative Clustering)", fontsize=14)
axes[0, 1].set_xlabel("UMAP Dimension 1")
axes[0, 1].set_ylabel("UMAP Dimension 2")
axes[0, 1].legend(title="Cluster ID")
# --- PLOT 3: Kohorten-Vergleich (Healthy vs Unhealthy) ---
# Zeigt, ob die ungesunden Patienten aus der Norm fallen.
axes[1, 0].scatter(
    healthy_umap[:, 0], healthy_umap[:, 1], 
    c="lightgrey", label="Healthy (SC)", s=30, alpha=0.5
)
if len(unhealthy_umap) > 0:
    axes[1, 0].scatter(
        unhealthy_umap[:, 0], unhealthy_umap[:, 1], 
        c="red", marker="X", s=60, label="Unhealthy (ST) Projected", alpha=0.8
    )
axes[1, 0].set_title("Projection of Unhealthy Cohort into Healthy Space", fontsize=14)
axes[1, 0].set_xlabel("UMAP Dimension 1")
axes[1, 0].set_ylabel("UMAP Dimension 2")
axes[1, 0].legend()
# --- PLOT 4: KLINISCHE BEDEUTUNG DER CLUSTER ---
# Dieser Plot ist der wichtigste für deine These! Er zeigt, WARUM sich Patienten unterscheiden.
df_melted = df_metrics.melt(id_vars=["Cluster"], value_vars=["% Wake", "% Light (N1+N2)", "% Deep (N3)", "% REM"], 
                            var_name="Sleep Stage", value_name="Percentage (%)")
sns.boxplot(
    data=df_melted, x="Sleep Stage", y="Percentage (%)", 
    hue="Cluster", palette="tab10", ax=axes[1, 1], showfliers=False
)
axes[1, 1].set_title("Clinical Meaning of Clusters (Sleep Stage Distribution)", fontsize=14)
axes[1, 1].set_ylabel("Percentage of Total Night (%)")
axes[1, 1].legend(title="Cluster ID")
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Platz für Suptitle lassen
plt.show()
